In [ ]:
transformer_part_of_day = spark.sql("""WITH hourly AS (
  SELECT 
      m.transformer_id,
      date_trunc('hour', r.timestamp) AS hour_ts,
      SUM(r.kwh) AS load_kwh
  FROM electric_raw_stage.transformer_meter_mapping m
  JOIN electric_raw_stage.meter_usage r
    ON m.meter_id = r.meter_id
  GROUP BY m.transformer_id, date_trunc('hour', r.timestamp)
),
daily AS (
  SELECT
      transformer_id,
      date_trunc('day', hour_ts) AS day_ts,

      SUM(CASE WHEN hour(hour_ts) BETWEEN 6 AND 11  THEN load_kwh ELSE 0 END) AS morning,
      SUM(CASE WHEN hour(hour_ts) BETWEEN 12 AND 17 THEN load_kwh ELSE 0 END) AS afternoon,
      SUM(CASE WHEN hour(hour_ts) BETWEEN 18 AND 23 THEN load_kwh ELSE 0 END) AS evening,
      SUM(CASE WHEN hour(hour_ts) BETWEEN 0 AND 5   THEN load_kwh ELSE 0 END) AS night

  FROM hourly
  GROUP BY transformer_id, date_trunc('day', hour_ts)
)

SELECT
    transformer_id,
    day_ts,
    ROUND(morning, 2)   AS morning,
    ROUND(afternoon, 2) AS afternoon,
    ROUND(evening, 2)   AS evening,
    ROUND(night, 2)     AS night,
    ROUND(morning + afternoon + evening + night, 2) AS total
FROM daily
ORDER BY transformer_id, day_ts""")

spark.sql("create database if not exists `electric_analysis_dev`")


transformer_part_of_day.coalesce(1).write \
    .mode("overwrite") \
    .format("parquet") \
    .option("path", "s3://ops-autopilot-data/transformed/transformer_parts_of_day/") \
    .saveAsTable("`electric_analysis_dev`.`transformer_parts_of_day_analysis`")